In [3]:
import os
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
path = os.path.join( "..", "data", "processed", "accidents_clean.csv")
accidents_clean = pd.read_csv(path)

In [ ]:
# On enlève les variables qui ont peu de poids (V Cramer <0.1) et celles qui ne sont pas informatives ou difficilement exploitables
# dans un modèle

accidents_clean = accidents_clean.drop(['id_vehicule','num_veh','jour','mois','an',
                                        'lum','int','atm','circ','prof','plan','catv',
                                        'surf','infra','lat', 'long'],axis=1)

In [9]:
import pandas as pd
import numpy as np
from itertools import combinations
from scipy.stats import chi2_contingency

def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    if confusion_matrix.shape[0] == 1 or confusion_matrix.shape[1] == 1:
        return np.nan  # Pas assez de variabilité
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    return np.sqrt(phi2 / min(k - 1, r - 1))

def analyser_categorielle_redundance(df, cat_vars, target):
    print("\n=== Association entre chaque variable catégorielle et la target ===")
    cramer_target = {}
    for var in cat_vars:
        v = cramers_v(df[var], df[target])
        cramer_target[var] = v
        print(f"Cramér’s V ({var} vs {target}) = {v:.3f}")

    print("\n=== Détection des redondances entre variables catégorielles ===")
    redondantes = []
    for var1, var2 in combinations(cat_vars, 2):
        v = cramers_v(df[var1], df[var2])
        if v > 0.5:  # Seuil à ajuster selon ton cas
            redondantes.append((var1, var2, v))
            print(f"⚠️ {var1} et {var2} sont très corrélées (Cramér’s V = {v:.3f})")

    if not redondantes:
        print("✅ Aucune paire fortement redondante trouvée.")

    return cramer_target, redondantes

In [10]:
cat_vars = ['catr', 'agg', 'col', 'obs','obsm','choc','manv','motor','place','catu','grav','sexe','trajet','secu1','dep','com','situ','vma','catv_regroupée']
target = 'grav'

analyser_categorielle_redundance(accidents_clean, cat_vars, target)


=== Association entre chaque variable catégorielle et la target ===
Cramér’s V (catr vs grav) = 0.108
Cramér’s V (agg vs grav) = 0.169
Cramér’s V (col vs grav) = 0.152
Cramér’s V (obs vs grav) = 0.160
Cramér’s V (obsm vs grav) = 0.145
Cramér’s V (choc vs grav) = 0.123
Cramér’s V (manv vs grav) = 0.163
Cramér’s V (motor vs grav) = 0.102
Cramér’s V (place vs grav) = 0.145
Cramér’s V (catu vs grav) = 0.173
Cramér’s V (grav vs grav) = 1.000
Cramér’s V (sexe vs grav) = 0.091
Cramér’s V (trajet vs grav) = 0.110
Cramér’s V (secu1 vs grav) = 0.271
Cramér’s V (dep vs grav) = 0.162
Cramér’s V (com vs grav) = 0.345
Cramér’s V (situ vs grav) = 0.131
Cramér’s V (vma vs grav) = 0.132
Cramér’s V (catv_regroupée vs grav) = 0.235

=== Détection des redondances entre variables catégorielles ===
⚠️ catr et agg sont très corrélées (Cramér’s V = 0.638)
⚠️ catr et com sont très corrélées (Cramér’s V = 0.530)
⚠️ agg et com sont très corrélées (Cramér’s V = 0.710)
⚠️ agg et vma sont très corrélées (Cramér’s 

({'catr': 0.10817823609051375,
  'agg': 0.16917623412172333,
  'col': 0.1521256305144429,
  'obs': 0.15972847114912153,
  'obsm': 0.14476099415191693,
  'choc': 0.12266885411497513,
  'manv': 0.16298064318752525,
  'motor': 0.10150954866135381,
  'place': 0.1447105501758004,
  'catu': 0.17311479682390404,
  'grav': 1.0,
  'sexe': 0.09147605593349231,
  'trajet': 0.10962034052388471,
  'secu1': 0.2709434775196114,
  'dep': 0.16191855072621214,
  'com': 0.34546798096607867,
  'situ': 0.13120840242608864,
  'vma': 0.13188928075879522,
  'catv_regroupée': 0.23537934136140934},
 [('catr', 'agg', 0.6378699325025724),
  ('catr', 'com', 0.530182126828249),
  ('agg', 'com', 0.710123094075433),
  ('agg', 'vma', 0.8503861961914926),
  ('place', 'catu', 0.9964187699131044),
  ('catu', 'secu1', 0.5365521667326909),
  ('dep', 'com', 0.9689577470818683),
  ('com', 'vma', 0.5891635850989145)])

In [11]:
# forte corrélation entre com d'un côté et catr, agg, dep et vma -> à supprimer
# vma correle aussi avec agg (agg plus simple + vma peut en partie être contenue dans catr même si corrélation < 0.5) -> à supprimer
# place correle avec catu (catu plus simple) -> à supprimer
# catu et secu 1 corrèlent -> Les piétons n'ont pas de dispo de sécurité. Je garderais quand même pour l'instant

accidents_clean = accidents_clean.drop(['com','vma', 'place'],axis=1)

In [14]:
# réencodage de la variable hrmn

accidents_clean['hrmn'] = pd.to_datetime(accidents_clean['hrmn'], format='%H:%M', errors='coerce')

# on récupère les heures
accidents_clean['heure'] = accidents_clean['hrmn'].dt.hour

# encodage cyclique sur les heures
accidents_clean['heure_sin'] = np.sin(2 * np.pi * accidents_clean['heure'] / 24)
accidents_clean['heure_cos'] = np.cos(2 * np.pi * accidents_clean['heure'] / 24)


# on supprime des variables intermédiaires et inutiles (jour, mois et an corrèlent peu avec grav -> on supprime date)
accidents_clean = accidents_clean.drop(['date','heure', 'hrmn'],axis=1)

In [20]:
accidents_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614559 entries, 0 to 614558
Data columns (total 20 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   obs             614559 non-null  int64  
 1   obsm            614559 non-null  int64  
 2   choc            614559 non-null  int64  
 3   manv            614559 non-null  int64  
 4   motor           614559 non-null  int64  
 5   catu            614559 non-null  int64  
 6   grav            614559 non-null  int64  
 7   sexe            614559 non-null  int64  
 8   trajet          614559 non-null  int64  
 9   secu1           614559 non-null  int64  
 10  dep             614559 non-null  object 
 11  agg             614559 non-null  int64  
 12  col             614559 non-null  int64  
 13  catr            614559 non-null  int64  
 14  nbv             614559 non-null  float64
 15  situ            614559 non-null  int64  
 16  age             614559 non-null  float64
 17  catv_regro

In [19]:
for col in accidents_clean.columns:
    print(f"{col}: {accidents_clean[col].nunique()} valeurs uniques")
    print(accidents_clean[col].unique())
    print("-" * 40)
accidents_clean.nunique()

obs: 18 valeurs uniques
[ 0  1  4 14  9  6 15 13  8  2 16 12  3  7 17 11  5 10]
----------------------------------------
obsm: 7 valeurs uniques
[2 0 1 9 6 4 5]
----------------------------------------
choc: 10 valeurs uniques
[5 3 1 4 2 0 8 6 7 9]
----------------------------------------
manv: 27 valeurs uniques
[23 11  0  2 21  1  9 26 15 17  4 12 16 19 13 14  3 10  5 24 18 20  7 22
 25  6  8]
----------------------------------------
motor: 7 valeurs uniques
[1 6 0 3 5 2 4]
----------------------------------------
catu: 3 valeurs uniques
[2 1 3]
----------------------------------------
grav: 4 valeurs uniques
[2 1 3 4]
----------------------------------------
sexe: 2 valeurs uniques
[2 1]
----------------------------------------
trajet: 7 valeurs uniques
[0 5 9 1 4 2 3]
----------------------------------------
secu1: 10 valeurs uniques
[1 2 8 0 3 4 5 6 9 7]
----------------------------------------
dep: 116 valeurs uniques
['93' '92' '94' '87' '69' '38' '34' '13' '988' '976' '974' '97

obs                18
obsm                7
choc               10
manv               27
motor               7
catu                3
grav                4
sexe                2
trajet              7
secu1              10
dep               116
agg                 2
col                 7
catr                8
nbv                13
situ                7
age               105
catv_regroupée      9
heure_sin          22
heure_cos          22
dtype: int64

In [ ]:
# dep et manv ont beaucoup de catégories -> prévilégier le target encoding
# obs : 18 catégories, à voir si target ou one one encoding
# les autres variables catégorielles : one hot encoding
# age et nbv numériques -> à normaliser ou standardiser
# heure_sin et heure_cos ok

# après l'encoding, on devrait avoir une petite centaine de variables ce qui devrait être gérable compte tenu des 600000+ entrées 